# House Price Prediction Model

A single walkthrough covering every Day 8 concept:
1. ML Lifecycle
2. Supervised Learning → Regression (price) + Classification (price tier)
3. Unsupervised Learning → Clustering (neighborhood segments)
4. Regression
5. Classification
6. Clustering overview
7. Train/Test split
8. Model evaluation metrics
9. Scikit-learn basics


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)


## Step 1 — ML Lifecycle: Problem Definition & Data Collection

Problem: Predict house sale price from property features.
Since no dataset was provided, we simulate a realistic one so the full pipeline can run end-to-end offline.

In [ ]:
N = 5000
area_sqft      = np.random.normal(1800, 600, N).clip(400, 5000)
bedrooms       = np.random.randint(1, 6, N)
bathrooms      = np.random.randint(1, 4, N)
age_years      = np.random.randint(0, 50, N)
location_score = np.random.uniform(1, 10, N)          # 10 = prime location
garage         = np.random.randint(0, 3, N)

noise = np.random.normal(0, 25000, N)
price = (
    area_sqft * 120
    + bedrooms * 8000
    + bathrooms * 6000
    - age_years * 900
    + location_score * 15000
    + garage * 5000
    + 20000
    + noise
).clip(50000, None)

df = pd.DataFrame({
    "area_sqft": area_sqft,
    "bedrooms": bedrooms,
    "bathrooms": bathrooms,
    "age_years": age_years,
    "location_score": location_score,
    "garage": garage,
    "price": price,
})

print(f"Dataset shape: {df.shape}")
df.head()


## Step 2 — Data Preprocessing (Scikit-learn basics: scaling)

In [ ]:
features = ["area_sqft", "bedrooms", "bathrooms", "age_years", "location_score", "garage"]
X = df[features]
y_price = df["price"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


## Step 3 — Regression + Train/Test Split (Supervised Learning)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_price, test_size=0.2, random_state=RANDOM_STATE
)
print(f"Train size: {X_train.shape[0]}  |  Test size: {X_test.shape[0]}")


In [ ]:
# Model A: Linear Regression
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)
y_pred_lin = lin_reg.predict(X_test)

# Model B: Random Forest Regressor
rf_reg = RandomForestRegressor(n_estimators=200, random_state=RANDOM_STATE)
rf_reg.fit(X_train, y_train)
y_pred_rf = rf_reg.predict(X_test)


## Step 4 — Model Evaluation Metrics (Regression)

In [ ]:
def regression_report(name, y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    mse = mean_squared_error(y_true, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_true, y_pred)
    print(f"[{name}]")
    print(f"  MAE  : {mae:,.2f}")
    print(f"  MSE  : {mse:,.2f}")
    print(f"  RMSE : {rmse:,.2f}")
    print(f"  R^2  : {r2:.4f}\n")
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

lin_metrics = regression_report("Linear Regression", y_test, y_pred_lin)
rf_metrics  = regression_report("Random Forest Regressor", y_test, y_pred_rf)

best_model_name = "Random Forest" if rf_metrics["R2"] > lin_metrics["R2"] else "Linear Regression"
print(f"Best regression model: {best_model_name}")


## Step 5 — Classification (Supervised Learning)

Bin price into Low/Medium/High tiers and predict the tier.

In [ ]:
tier_labels = pd.qcut(df["price"], q=3, labels=["Low", "Medium", "High"])
df["price_tier"] = tier_labels

Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X_scaled, tier_labels, test_size=0.2, random_state=RANDOM_STATE, stratify=tier_labels
)

clf = LogisticRegression(max_iter=1000)
clf.fit(Xc_train, yc_train)
yc_pred = clf.predict(Xc_test)


In [ ]:
acc = accuracy_score(yc_test, yc_pred)
prec = precision_score(yc_test, yc_pred, average="macro")
rec = recall_score(yc_test, yc_pred, average="macro")
f1 = f1_score(yc_test, yc_pred, average="macro")
print(f"Accuracy : {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall   : {rec:.4f}")
print(f"F1-score : {f1:.4f}")

cm = confusion_matrix(yc_test, yc_pred, labels=["Low", "Medium", "High"])
print("Confusion Matrix (rows=actual, cols=predicted, order=Low/Medium/High):")
print(cm)


## Step 6 — Unsupervised Learning: Clustering (Neighborhood Segments)

In [ ]:
kmeans = KMeans(n_clusters=3, random_state=RANDOM_STATE, n_init=10)
df["cluster"] = kmeans.fit_predict(X_scaled)
df.groupby("cluster")[features + ["price"]].mean().round(1)


## Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

best_pred = y_pred_rf if best_model_name == "Random Forest" else y_pred_lin
axes[0].scatter(y_test, best_pred, alpha=0.4, color="#2563eb")
axes[0].plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], "r--")
axes[0].set_xlabel("Actual Price")
axes[0].set_ylabel("Predicted Price")
axes[0].set_title(f"Actual vs Predicted ({best_model_name})")

importances = pd.Series(rf_reg.feature_importances_, index=features).sort_values()
axes[1].barh(importances.index, importances.values, color="#16a34a")
axes[1].set_title("Feature Importance (Random Forest)")

scatter = axes[2].scatter(df["area_sqft"], df["price"], c=df["cluster"], cmap="viridis", alpha=0.6)
axes[2].set_xlabel("Area (sqft)")
axes[2].set_ylabel("Price")
axes[2].set_title("Clusters: Area vs Price")

plt.tight_layout()
plt.show()


## Summary

- **Regression**: predicted exact house price (MAE, MSE, RMSE, R²)
- **Classification**: predicted price tier — Low / Medium / High (Accuracy, Precision, Recall, F1, Confusion Matrix)
- **Clustering**: segmented houses into 3 unsupervised neighborhood groups
- **Next steps**: hyperparameter tuning, cross-validation, real-world dataset, deployment & monitoring